In [1]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import xml.etree.ElementTree as ET
from tqdm.notebook import tqdm
import torch.nn.functional as F

# --- CONFIGURATION ---
CONFIG = {
    'xml_root': r'E:\DATA\Annotations',
    'video_root': r'E:\DATA\Videos', 
    
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'target_label': 'bangla-tesla', 
    'obs_len': 15,  
    'pred_len': 45, 
    
    # BiTraP Specifics
    'hidden_size': 256,
    'embed_size': 64,
    'latent_size': 32,    # Size of the "random" noise vector z
    'kld_weight': 0.05,   # Weight for the KL Divergence loss
    
    'batch_size': 32,
    'epochs': 20,
    'lr': 1e-3
}

print(f"✅ BiTraP Config Loaded. Device: {CONFIG['device']}")

✅ BiTraP Config Loaded. Device: cpu


In [2]:
class IDDTrajectoryDataset(Dataset):
    def __init__(self, xml_root, obs_len=15, pred_len=45, target_label='bangla-tesla'):
        self.obs_len = obs_len
        self.pred_len = pred_len
        self.seq_len = obs_len + pred_len
        self.samples = []
        
        print(f"📂 Parsing XMLs for label: '{target_label}'...")
        xml_files = glob.glob(os.path.join(xml_root, '**', '*.xml'), recursive=True)
        
        for xml in tqdm(xml_files):
            try:
                tree = ET.parse(xml)
                root = tree.getroot()
                meta_size = root.find('meta').find('original_size')
                img_w = float(meta_size.find('width').text)
                img_h = float(meta_size.find('height').text)
                
                for track in root.findall('track'):
                    if track.attrib['label'] != target_label: continue
                    track_data = [] 
                    boxes = sorted(track.findall('box'), key=lambda b: int(b.attrib['frame']))
                    for box in boxes:
                        if box.get('outside') == '1': continue
                        xtl, ytl = float(box.attrib['xtl']), float(box.attrib['ytl'])
                        xbr, ybr = float(box.attrib['xbr']), float(box.attrib['ybr'])
                        w = xbr - xtl
                        h = ybr - ytl
                        cx = (xtl + xbr) / 2
                        cy = (ytl + ybr) / 2
                        track_data.append([cx/img_w, cy/img_h, w/img_w, h/img_h])
                    
                    track_data = np.array(track_data)
                    if len(track_data) < self.seq_len: continue
                    
                    stride = 10 
                    for i in range(0, len(track_data) - self.seq_len + 1, stride):
                        obs = track_data[i : i+obs_len]
                        pred = track_data[i+obs_len : i+obs_len+pred_len]
                        self.samples.append({
                            'obs': obs[:, 0:2], # Input: cx, cy
                            'pred': pred        # Target: cx, cy, w, h
                        })
            except: pass

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        item = self.samples[idx]
        return (
            torch.tensor(item['obs'], dtype=torch.float32), 
            torch.tensor(item['pred'], dtype=torch.float32)
        )

# Create Dataset
dataset = IDDTrajectoryDataset(CONFIG['xml_root'], obs_len=CONFIG['obs_len'], pred_len=CONFIG['pred_len'])
print(f"✅ Dataset Created: {len(dataset)} sequences.")

📂 Parsing XMLs for label: 'bangla-tesla'...


  0%|          | 0/6 [00:00<?, ?it/s]

✅ Dataset Created: 2156 sequences.


In [3]:
class BiTraP(nn.Module):
    def __init__(self, input_size=2, output_size=4, hidden_size=256, embed_size=64, latent_size=32):
        super(BiTraP, self).__init__()
        
        # 1. Coordinate Embedding
        self.embed = nn.Linear(input_size, embed_size)
        self.relu = nn.ReLU()
        
        # 2. Bi-Directional Encoder (History)
        # Output dim will be hidden_size * 2
        self.encoder_obs = nn.LSTM(embed_size, hidden_size, batch_first=True, bidirectional=True)
        
        # 3. Future Encoder (Only used during training for Posterior)
        # Input is 4 (x,y,w,h) because target has dimensions
        self.embed_fut = nn.Linear(output_size, embed_size)
        self.encoder_fut = nn.LSTM(embed_size, hidden_size, batch_first=True, bidirectional=True)
        
        # 4. Latent Space (CVAE)
        # Encoder Hidden State (bidirectional) -> 2 * hidden
        enc_out_dim = hidden_size * 2
        
        # Prior Network P(z|X) (Used at Inference)
        self.prior_mu = nn.Linear(enc_out_dim, latent_size)
        self.prior_logvar = nn.Linear(enc_out_dim, latent_size)
        
        # Posterior Network Q(z|X,Y) (Used at Training)
        # Takes History + Future features
        self.post_mu = nn.Linear(enc_out_dim * 2, latent_size)
        self.post_logvar = nn.Linear(enc_out_dim * 2, latent_size)
        
        # 5. Decoder
        # Input: Embedding + Latent z
        self.decoder_lstm = nn.LSTM(embed_size + latent_size, hidden_size, batch_first=True)
        self.out_fc = nn.Linear(hidden_size, output_size)
        
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
        
    def forward(self, obs, pred_len, future=None):
        # obs: [Batch, 15, 2]
        # future: [Batch, 45, 4] (Optional, only for training)
        
        # --- ENCODE HISTORY ---
        obs_emb = self.relu(self.embed(obs))
        # h_obs shape: [2, Batch, Hidden] (2 for bidirection)
        # output shape: [Batch, Seq, 2*Hidden]
        _, (h_obs, _) = self.encoder_obs(obs_emb)
        
        # Concatenate forward and backward hidden states
        # h_obs is [2, B, H]. We want [B, 2*H]
        state_obs = torch.cat((h_obs[0], h_obs[1]), dim=1) 
        
        # --- CVAE LOGIC ---
        if future is not None:
            # Training Mode: Encode Future + History -> Posterior
            fut_emb = self.relu(self.embed_fut(future))
            _, (h_fut, _) = self.encoder_fut(fut_emb)
            state_fut = torch.cat((h_fut[0], h_fut[1]), dim=1)
            
            # Fuse History and Future
            state_total = torch.cat([state_obs, state_fut], dim=1)
            
            # Calculate Posterior (Q)
            mu = self.post_mu(state_total)
            logvar = self.post_logvar(state_total)
            z = self.reparameterize(mu, logvar)
            
            # Also calculate Prior (P) to compute KL Divergence loss
            prior_mu = self.prior_mu(state_obs)
            prior_logvar = self.prior_logvar(state_obs)
            
        else:
            # Inference Mode: History -> Prior
            mu = self.prior_mu(state_obs)
            logvar = self.prior_logvar(state_obs)
            z = self.reparameterize(mu, logvar) # Sample from Prior
            
            prior_mu, prior_logvar = mu, logvar # For return consistency
            
        # --- DECODER ---
        outputs = []
        curr_input = obs[:, -1, :].unsqueeze(1) # [Batch, 1, 2]
        
        # Init decoder hidden state (Zero init is standard, or project z)
        h_dec = torch.zeros(1, obs.size(0), CONFIG['hidden_size']).to(obs.device)
        c_dec = torch.zeros(1, obs.size(0), CONFIG['hidden_size']).to(obs.device)
        
        for _ in range(pred_len):
            # Input is: Current Coordinate Embedding + Latent z (concatenated)
            curr_emb = self.relu(self.embed(curr_input)) # [Batch, 1, Embed]
            
            # Expand z to match sequence length 1
            z_expanded = z.unsqueeze(1) # [Batch, 1, Latent]
            
            dec_input = torch.cat([curr_emb, z_expanded], dim=2)
            
            out, (h_dec, c_dec) = self.decoder_lstm(dec_input, (h_dec, c_dec))
            
            pred_step = self.out_fc(out) # [Batch, 1, 4]
            outputs.append(pred_step)
            
            curr_input = pred_step[:, :, :2] # Feed back [cx, cy]
            
        pred_traj = torch.cat(outputs, dim=1)
        
        return pred_traj, mu, logvar, prior_mu, prior_logvar

# Initialize
model = BiTraP(
    input_size=2, 
    output_size=4,
    hidden_size=CONFIG['hidden_size'],
    embed_size=CONFIG['embed_size'],
    latent_size=CONFIG['latent_size']
).to(CONFIG['device'])

print(f"✅ BiTraP Model Initialized.")

✅ BiTraP Model Initialized.


In [4]:
# 1. Split Data
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_set, batch_size=CONFIG['batch_size'], shuffle=False)

optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])

def cvae_loss(pred, target, mu, logvar, prior_mu, prior_logvar):
    # 1. Reconstruction Loss (MSE)
    mse = F.mse_loss(pred, target, reduction='mean')
    
    # 2. KL Divergence
    # Measure difference between Posterior (mu, logvar) and Prior (prior_mu, prior_logvar)
    # If Inference, Posterior = Prior, so KLD is 0.
    
    # Formula for KLD between two Gaussians
    kld = 0.5 * torch.mean(
        prior_logvar - logvar - 1 
        + (logvar.exp() + (mu - prior_mu).pow(2)) / prior_logvar.exp()
    )
    
    return mse + (CONFIG['kld_weight'] * kld), mse, kld

print(f"🚀 Starting BiTraP Training...")

for epoch in range(CONFIG['epochs']):
    model.train()
    total_loss = 0
    total_mse = 0
    total_kld = 0
    
    for obs, target in tqdm(train_loader, leave=False, desc=f"Epoch {epoch+1}"):
        obs = obs.to(CONFIG['device'])
        target = target.to(CONFIG['device'])
        
        optimizer.zero_grad()
        
        # Pass 'target' (future) to use Posterior during training
        preds, mu, logvar, p_mu, p_logvar = model(obs, CONFIG['pred_len'], future=target)
        
        loss, mse, kld = cvae_loss(preds, target, mu, logvar, p_mu, p_logvar)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_mse += mse.item()
        total_kld += kld.item()
        
    # Validation (Inference Mode: future=None)
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for obs, target in val_loader:
            obs = obs.to(CONFIG['device'])
            target = target.to(CONFIG['device'])
            
            # No future passed -> Uses Prior
            preds, mu, logvar, p_mu, p_logvar = model(obs, CONFIG['pred_len'], future=None)
            
            # Loss against ground truth
            loss, _, _ = cvae_loss(preds, target, mu, logvar, p_mu, p_logvar)
            val_loss += loss.item()
            
    avg_train = total_loss / len(train_loader)
    avg_mse = total_mse / len(train_loader)
    avg_val = val_loss / len(val_loader)
    
    print(f"Epoch {epoch+1} | Loss: {avg_train:.5f} (MSE: {avg_mse:.5f}) | Val: {avg_val:.5f}")

torch.save(model.state_dict(), "bangla_tesla_bitrap.pth")
print("💾 BiTraP Model Saved.")

🚀 Starting BiTraP Training...


Epoch 1:   0%|          | 0/54 [00:00<?, ?it/s]

Epoch 1 | Loss: 0.02409 (MSE: 0.02358) | Val: 0.01217


Epoch 2:   0%|          | 0/54 [00:00<?, ?it/s]

Epoch 2 | Loss: 0.00862 (MSE: 0.00777) | Val: 0.00610


...

Epoch 18 | Loss: 0.00251 (MSE: 0.00184) | Val: 0.00383


Epoch 19:   0%|          | 0/54 [00:00<?, ?it/s]

Epoch 19 | Loss: 0.00235 (MSE: 0.00169) | Val: 0.00386


Epoch 20:   0%|          | 0/54 [00:00<?, ?it/s]

Epoch 20 | Loss: 0.00225 (MSE: 0.00165) | Val: 0.00398
💾 BiTraP Model Saved.


In [5]:
import pickle 

def calculate_bitrap_metrics(model, loader):
    model.eval()
    mse_traj_list, cmse_final_list, cfmse_final_list = [], [], []
    W_px, H_px = 2592, 1944 
    
    with torch.no_grad():
        for obs, target in loader:
            obs = obs.to(CONFIG['device'])
            target = target.cpu().numpy()
            
            # Inference: future=None (uses Prior)
            # For CVAE, you can sample multiple times (Best-of-N), 
            # but for basic comparison we take one sample (or the mean).
            preds, _, _, _, _ = model(obs, CONFIG['pred_len'], future=None)
            preds = preds.cpu().numpy()
            
            # --- Un-normalize ---
            pred_seq = np.zeros_like(preds)
            gt_seq = np.zeros_like(target)
            
            pred_seq[:, :, 0] = preds[:, :, 0] * W_px; pred_seq[:, :, 2] = preds[:, :, 2] * W_px
            pred_seq[:, :, 1] = preds[:, :, 1] * H_px; pred_seq[:, :, 3] = preds[:, :, 3] * H_px
            gt_seq[:, :, 0] = target[:, :, 0] * W_px; gt_seq[:, :, 2] = target[:, :, 2] * W_px
            gt_seq[:, :, 1] = target[:, :, 1] * H_px; gt_seq[:, :, 3] = target[:, :, 3] * H_px
            
            # --- Metrics ---
            # MSE
            traj_mse = np.mean(np.sum((pred_seq[:,:,:2] - gt_seq[:,:,:2])**2, axis=2), axis=1)
            mse_traj_list.extend(traj_mse)

            # C-MSE
            c_mse = np.sum((pred_seq[:, -1, :2] - gt_seq[:, -1, :2])**2, axis=1)
            cmse_final_list.extend(c_mse)
            
            # CF-MSE
            gt_foot = np.stack([gt_seq[:, -1, 0], gt_seq[:, -1, 1] + gt_seq[:, -1, 3]/2], axis=1)
            pred_foot = np.stack([pred_seq[:, -1, 0], pred_seq[:, -1, 1] + pred_seq[:, -1, 3]/2], axis=1)
            foot_mse = np.sum((pred_foot - gt_foot)**2, axis=1)
            cfmse_final_list.extend(c_mse + foot_mse)

    return np.mean(mse_traj_list), np.mean(cmse_final_list), np.mean(cfmse_final_list)

print("📊 Calculating BiTraP Metrics...")
mse, c_mse, cf_mse = calculate_bitrap_metrics(model, val_loader)

print(f"\n✅ BiTraP Results (Bangla-Tesla):")
print(f"   MSE (Avg Trajectory): {mse:.2f}")
print(f"   C-MSE (Center @ 1.5s):  {c_mse:.2f}")
print(f"   CF-MSE (Center+Foot @ 1.5s): {cf_mse:.2f}")

with open('result_bangla_tesla_bitrap.pkl', 'wb') as f:
    pickle.dump({'MSE': mse, 'C-MSE': c_mse, 'CF-MSE': cf_mse}, f)

📊 Calculating BiTraP Metrics...

✅ BiTraP Results (Bangla-Tesla):
   MSE (Avg Trajectory): 20849.88
   C-MSE (Center @ 1.5s):  50561.20
   CF-MSE (Center+Foot @ 1.5s): 112180.95
